In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Display initial information
print("Data Shape:", df.shape)
print("\nData Types and Non-Null Counts:")
df.info()

print("\nFirst 5 rows:")
print(df.head())

In [ ]:
# Convert Total_Charges to numeric, handling non-numeric entries (likely spaces)
df['Total_Charges'] = pd.to_numeric(df['Total_Charges'], errors='coerce')

# Check for missing values in Total_Charges
missing_total_charges = df['Total_Charges'].isnull().sum()
print(f"\nNumber of missing values in Total_Charges: {missing_total_charges}")

# For a small number of missing values (if any), a good imputation strategy is to replace
# them with the median, especially since these cases often correspond to customers with 0 tenure.
# Customers with 0 tenure likely just signed up, so their Total_Charges should be close to Monthly_Charges.
# However, for simplicity, I will fill with the median for now, or drop them if the count is very low.
# Given the context (Telco Churn), these are likely new customers, so imputing with the median is safer than dropping.
df['Total_Charges'].fillna(df['Total_Charges'].median(), inplace=True)
print("Missing Total_Charges handled by median imputation.")

In [ ]:
# Convert 'Churn' to a binary numeric column (0 and 1)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
churn_counts = df['Churn'].value_counts(normalize=True) * 100

print("\nChurn Distribution:")
print(churn_counts)

# (Visualization would show a significant class imbalance: ~73.5% 'No', ~26.5% 'Yes')

In [ ]:
numerical_cols = ['tenure', 'Monthly_Charges', 'Total_Charges']
print("\nNumerical Feature Statistics:")
print(df[numerical_cols].describe())

# (Visualization would show:
# 1. tenure: Bimodal distribution, with peaks at very low and very high values.
# 2. Monthly_Charges: Skewed towards lower values, but also a cluster around high values (Fiber Optic users).
# 3. Total_Charges: Highly skewed, as expected, since it's cumulative.
# No severe outliers are apparent that would require aggressive clipping, but scaling is essential.)

In [ ]:
# Identify categorical columns (excluding 'customerID' and numeric ones)
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print("\nUnique values in key categorical columns:")
for col in ['gender', 'Dependents', 'Phone_Service', 'Internet_Service', 'Contract']:
    print(f"  {col}: {df[col].unique()}")

# (Visualization would show:
# - Contract: Month-to-month has a much higher churn rate.
# - Internet_Service: Fiber optic users churn significantly more.
# - Payment_Method: Electronic check has the highest churn rate.
# - Online_Security/Tech_Support: Lack of these services increases churn.)

In [ ]:
# Drop the identifier column
df.drop('customerID', axis=1, inplace=True)
print("\nDropped 'customerID'.")

In [ ]:
print(df.columns.to_list())

In [ ]:
# Re-map 'No phone service' and 'No internet service' to 'No'
for col in df.columns:
    if 'No phone service' in df[col].unique():
        df[col] = df[col].replace('No phone service', 'No')
    if 'No internet service' in df[col].unique():
        df[col] = df[col].replace('No internet service', 'No')

# Label Encode binary columns (Yes/No and gender)
binary_cols = [
    'Is_Married', 'Dependents', 'Phone_Service', 'Paperless_Billing',
    'Dual', 'Online_Security', 'Online_Backup', 'Device_Protection',
    'Tech_Support', 'Streaming_TV', 'Streaming_Movies'
]
# For gender (Male/Female)
df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})

# For Yes/No columns
for col in binary_cols[2:]:
    # Ensure all are Yes/No before mapping (SeniorCitizen is already 0/1)
    if df[col].dtype == 'object':
        df[col] = df[col].map({'No': 0, 'Yes': 1})

print("\nBinary columns label-encoded (Yes/No to 1/0, Female/Male to 0/1).")

In [ ]:
# change Dependents to 0/1
df['Dependents'] = df['Dependents'].map({'No': 0, 'Yes': 1})
df['Is_Married'] = df['Is_Married'].map({'No': 0, 'Yes': 1})

In [ ]:
# Identify columns for One-Hot Encoding
ohe_cols = ['Internet_Service', 'Contract', 'Payment_Method']

# Apply One-Hot Encoding
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)
print("Multinomial columns One-Hot Encoded (drop_first=True).")

# Prepare feature matrix X and target vector y
X = df.drop('Churn', axis=1)
y = df['Churn']

In [ ]:
# Identify numerical columns for scaling
numerical_cols = ['tenure', 'Monthly_Charges', 'Total_Charges']

# Initialize StandardScaler
scaler = StandardScaler()

# Fit and transform the numerical features
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print("\nNumerical features scaled using StandardScaler.")
print("\nFinal Transformed Feature Matrix (X) Head:")
print(X.head())
print(f"Final X shape: {X.shape}")

In [ ]:
# Save the transformed data to a CSV for completeness
X_y = X.copy()
X_y['Churn'] = y
X_y.to_csv('telco_churn_transformed_for_svm.csv', index=False)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Load the transformed data
df_transformed = pd.read_csv('telco_churn_transformed_for_svm.csv')

# 2. Separate features (X) and target (y)
X = df_transformed.drop('Churn', axis=1)
y = df_transformed['Churn']

# 3. Split the data into training and testing sets
# Use a stratify approach to maintain the class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


In [ ]:
X_train.dtypes

In [ ]:

# 4. Define the parameter grid for GridSearch
# Using a small, representative grid for quick computation and demonstration
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': [1, 0.1, 0.01],
    'kernel': ['rbf']
}

# 5. Initialize the SVM model with class_weight='balanced'
# This is crucial for addressing the class imbalance found in the EDA.
svc = SVC(class_weight='balanced', random_state=42)

# 6. Initialize GridSearchCV
# Use 'f1' as the scoring metric since the data is imbalanced and we care about both precision and recall for the positive class (Churn).
grid = GridSearchCV(svc, param_grid, refit=True, verbose=2, cv=3, scoring='f1', n_jobs=-1)

# 7. Fit the model to the training data
grid.fit(X_train, y_train)

# 8. Report the best parameters and score
print("\n--- Best Parameters and Score ---")
print(f"Best Parameters: {grid.best_params_}")
print(f"Best F1-Score (Cross-Validation): {grid.best_score_:.4f}")

# 9. Evaluate the best model on the test set
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\n--- Model Evaluation on Test Set ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Print confusion matrix for detailed performance view
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix (Test Set):")
print(cm)